# CS 3892 / 5892 — Session 18 · Loop invariants and abstract interpretation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-29-loop-invariants-and-abstract-interpretation.ipynb)

**Thursday, October 29, 2026.** CBMC answers for runs up to a bound. Two ways past the bound:
**deductive verification**, where a loop invariant meeting three conditions proves the loop for every
number of iterations; and **abstract interpretation**, which computes over abstract values (signs,
intervals) instead of concrete ones, always terminates, and is sound but imprecise. Plus two things
CBMC does besides verification: generating tests, and exposing **vacuous** specifications.

Every file asserts its own results. `absint.py` is a small abstract interpreter written for this
session. CBMC (free, BSD) is fetched by the setup cell.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, fetch CBMC, define helpers
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-29-loop-invariants-and-abstract-interpretation"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

# Colab does not ship Z3; install it if this machine lacks it.
try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

PYD = ROOT / "sessions" / SESSION / "python"
CD  = ROOT / "sessions" / SESSION / "c"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- CBMC: the official 6.11.0 release, fetched once (about 70 MB) and checked by SHA-256.
# cbmc_cli.py unpacks the Ubuntu 22.04 package in Python: no root, no dpkg needed.
out = run(PYD / "cbmc_cli.py")
assert "CBMC 6.11.0" in out

## 1. Generating tests with a model checker

`cbmc --cover branch` turns each branch into a goal and prints one input per goal.

In [ ]:
show(CD / "shipping.c")
out = run(PYD / "01_test_generation.py")
assert "100.0%" in out

## 2. A loop invariant, and its three conditions

Established before the loop, preserved by one iteration, sufficient for the postcondition on exit.
Each condition is a formula; Z3 checks it.

In [ ]:
out = run(PYD / "02_division_vcs.py")
assert "3 sufficient   FAILS" in out

## 3. The same conditions on a C loop: CBMC loop contracts

Session 9's add-2 counter as a loop. The property itself is not inductive; the strengthening is.

In [ ]:
display(SVG(filename=str(FIG / "add2.svg")))
show(CD / "add2_loop.c")
out = run(PYD / "03_loop_contract.py")
assert "counterexample to induction: x =" in out and "VERIFICATION SUCCESSFUL" in out

## 4. Abstract domains: signs and intervals

`absint.py` holds the two domains and the interpreter. What can each say, and what does a join lose?

In [ ]:
show(PYD / "absint.py")
out = run(PYD / "04_signs_and_intervals.py")
assert "y = top" in out and "y = [-2, 2]" in out

## 5. The running counter: the fixpoint, widening, narrowing

Plain join reaches `[0, 10]` in 11 rounds, the same count as session 16's BDD fixpoint. Widening is
faster and loses `x <= 10`; one narrowing step recovers it -- the invariant from session 9, found by
the tool.

In [ ]:
display(SVG(filename=str(FIG / "counter.svg")))
out = run(PYD / "05_widening.py")
assert "x is in [0, 10]" in out and "gives [0, 10]" in out

## 6. What a join loses: a false alarm, and what a relational domain keeps

In [ ]:
show(CD / "correlated.c")
out = run(PYD / "06_join_loses_correlation.py")
assert "UNKNOWN" in out and "SUCCESS" in out

## 7. Vacuous specifications: "verified", and saying nothing

In [ ]:
show(CD / "vacuous.c")
out = run(PYD / "07_vacuity.py")
assert "VERIFICATION SUCCESSFUL" in out and "satisfiable? unsat" in out

## Try it yourself

- In `02`, try `Inv: x = q*y + r` without `r >= 0`. Which condition fails, and why?
- In `03`, try the invariant `x % 2 == 0` with no bounds. What does CBMC report, and why does the cap
  matter?
- In `05`, delete the reset (make the else branch `x := x + 1` too). Does plain join stop now? Does
  widening? What does narrowing give?
- In `06`, change the else branch to `x := 2; y := 3`. Is there a real division by zero now? What do
  intervals, the relational view and CBMC each say?